# 4장 — 몸집과 밥의 황금비: Chinchilla 🐭

같은 예산 C = 6·N·D 로 **"크게 잠깐" vs "작게 오래"**, 어느 배분이 이길까요?
3장의 법칙을 N축·D축에 하나씩 얹으면 이 질문은 계산이 됩니다:

$$L(N, D) = E + \frac{A}{\sqrt{N}} + \frac{B}{\sqrt{D}}$$

상수(E=1.69, A=32,000, B=A·√20)는 친칠라의 실제 관측(**70B·1.4T가 최적, 파라미터당
약 20토큰, 예산 100배면 N·D 각 10배**)을 재현하도록 고른 단순화 모형입니다
(실제 논문의 지수는 α≈0.34, β≈0.28로 비대칭. 읽기: **알파·베타**, 두 배고픔이 줄어드는 속도의
지수. 정직한 각주는 웹 챕터 더 깊이 카드).

> 최저점 유도 감 잡기: 두 배고픔이 같아지는 곳 A/√N = B/√D 에서 **양변을 제곱**하면
> A²/N = B²/D → D* = (B/A)²·N* = **20·N***. 아래 코드의 `optimum()` 이 바로 이 식입니다.

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 여기서 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch4')

## 1. 예산 고정 배분 곡선: U자와 최저점

예산을 친칠라급(C = 5.88×10²³)으로 고정하고 몸집 N만 바꿔 봅니다. D = C/(6N)은 자동.

In [ ]:
import math
import numpy as np

E, A = 1.69, 32000.0
B = A * math.sqrt(20)

def loss(n, d):
    return E + A / math.sqrt(n) + B / math.sqrt(d)

def optimum(c):
    """최저점 — dL/dN = 0 을 풀면 N* = (A/B)·√(C/6), D* = 20·N*."""
    K = c / 6
    n = (A / B) * math.sqrt(K)
    return n, K / n

C = 5.88e23  # 친칠라급 예산
n_star, d_star = optimum(C)
print(f'최적 배분: N* = {n_star/1e9:.1f}B, D* = {d_star/1e12:.2f}T, 비율 = {d_star/n_star:.1f} 토큰/파라미터')
print(f'최저 손실: L* = {loss(n_star, d_star):.4f}')
print(f'최적점에서 두 배고픔이 같다: A/√N* = {A/math.sqrt(n_star):.4f}, B/√D* = {B/math.sqrt(d_star):.4f}')

assert abs(n_star - 70e9) / 70e9 < 1e-9, '친칠라의 70B가 정확히 최적'
assert abs(d_star - 1.4e12) / 1.4e12 < 1e-9, '1.4T 토큰'
assert abs(d_star / n_star - 20) < 1e-9, '황금비 20'
assert abs(loss(n_star, d_star) - 1.9319) < 1e-4
assert abs(A / math.sqrt(n_star) - B / math.sqrt(d_star)) < 1e-12, '두 항 동일'
print()
print('✔ 친칠라 관측 재현: 70B·1.4T, 황금비 20, 두 배고픔의 균형점')

## 2. 역사의 재현: 고퍼(280B) vs 친칠라(70B), 같은 예산

"몸집이 깡패" 시절의 배분(280B에 350B 토큰)과 황금비 배분을 같은 예산에서 비교합니다.

In [ ]:
cases = [
    ('고퍼 스타일   280B', 280e9),
    ('친칠라 스타일  70B', 70e9),
]
print('배분              |     D     | 토큰/파라미터 | 손실 L')
print('-' * 60)
for name, n in cases:
    d = C / 6 / n
    print(f'{name} | {d/1e12:5.2f}T | {d/n:11.2f} | {loss(n, d):.4f}')

L_gopher = loss(280e9, C / 6 / 280e9)
L_chin = loss(70e9, C / 6 / 70e9)
assert abs(L_gopher - 1.9924) < 1e-4 and abs(L_chin - 1.9319) < 1e-4
assert L_chin < L_gopher, '4배 작은 모델이 같은 예산으로 이긴다'

# GPT-3 (자기 예산 3.15e23, 175B·300B — 비율 1.7)
C3 = 6 * 175e9 * 300e9
n3s, d3s = optimum(C3)
print()
print(f'GPT-3 스타일: 175B·0.30T (비율 1.7) → L = {loss(175e9, 300e9):.4f}')
print(f'같은 예산 최적: {n3s/1e9:.0f}B·{d3s/1e12:.2f}T → L = {loss(n3s, d3s):.4f}')
assert abs(loss(175e9, 300e9) - 2.0278) < 1e-4 and abs(loss(n3s, d3s) - 1.9727) < 1e-4
print()
print('✔ 과체격(비율 1~2)의 손해 실측. 같은 돈으로 더 잘할 수 있었다')

## 3. 그림: U자 곡선과 반반 성장

왼쪽: 예산별 배분 곡선(최저점이 전부 비율 20). 오른쪽: 예산 100배 → 최적 N*·D* 각 10배.

In [ ]:
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False
except Exception:
    pass

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# (좌) U자 곡선 — 예산 3종
for c, color in [(1e22, '#94a3b8'), (5.88e23, '#6366f1'), (1e25, '#0f172a')]:
    ns, _ = optimum(c)
    offs = np.linspace(-2, 2, 200)
    Ls = [loss(ns * 10 ** o, c / 6 / (ns * 10 ** o)) for o in offs]
    ax1.plot(offs, Ls, color=color, label=f'C = {c:.0e}')
    ax1.plot(0, loss(*optimum(c)), 'o', color='#10b981')
ax1.plot(np.log10(280e9 / n_star), L_gopher, 's', color='#e11d48', ms=8, label='고퍼 스타일')
ax1.set_xlabel('배분 (log10 N − log10 N*)')
ax1.set_ylabel('손실 L')
ax1.set_title('예산 고정 배분 곡선: 최저점 = 황금비')
ax1.legend(fontsize=8)

# (우) 예산 vs 최적 N*, D* (로그-로그 — 기울기 1/2 = 반반 성장)
cs = np.logspace(21, 25, 20)
ns = [optimum(c)[0] for c in cs]
ds = [optimum(c)[1] for c in cs]
ax2.loglog(cs, ns, '-', color='#6366f1', label='최적 몸집 N*')
ax2.loglog(cs, ds, '-', color='#10b981', label='최적 밥 D*')
ax2.set_xlabel('예산 C (FLOPs)')
ax2.set_title('예산 100배 → N*·D* 각 10배 (반반 성장)')
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

n1, d1 = optimum(1e22)
n2, d2 = optimum(1e24)
assert abs(n2 / n1 - 10) < 1e-9 and abs(d2 / d1 - 10) < 1e-9
print(f'✔ 반반 성장: C 10²²→10²⁴ (100배) 에서 N* {n1/1e9:.1f}B→{n2/1e9:.1f}B, D* {d1/1e12:.2f}T→{d2/1e12:.2f}T')

## 4. 황금비 검산: 웹 StepQuiz 의 그 문제

예산 C = 6×10²² 를 황금비(D = 20N)로 배분하면? C = 6·N·20N = **120·N²**.

In [ ]:
C_quiz = 6e22
N_quiz = math.sqrt(C_quiz / 120)
D_quiz = 20 * N_quiz
print(f'N = √({C_quiz/120:.0e}) = {N_quiz/1e9:.1f}B,  D = {D_quiz/1e9:.0f}B 토큰')
assert abs(N_quiz - math.sqrt(5e20)) < 1
assert round(N_quiz / 1e9) == 22
# 검산: 이 배분이 정말 최적인가
ns_q, ds_q = optimum(C_quiz)
assert abs(ns_q - N_quiz) / N_quiz < 1e-12
print('✔ 공식 최적과 일치. 예산 하나로 설계(N, D)가 통째로 나온다')

## 정리: 오늘 가져갈 세 문장

1. 같은 예산이라도 배분에 따라 손실이 다르다: **U자 곡선**, 오른쪽이 과체격(덜 먹임).
2. 최저점 = 두 배고픔(A/√N, B/√D)의 균형점, 어림 규칙 **파라미터당 약 20토큰**,
   예산이 늘면 **몸집과 밥을 반반**(각 √배)으로 키운다.
3. 요즘은 서빙 비용 때문에 황금비를 일부러 넘겨 먹이기도 한다(8B·15T = 1,875:1).
   비율 하나로 **설계 의도**까지 읽힌다.

다음 **5장(창발일까 착시일까)**: "능력이 갑자기 튀어나온다"는 주장을 검증대에 올립니다.